# Создание датасета

In [ ]:
import sys
import unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import torch
import tqdm

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent))

from shared import config

print(f"DATA_DIR: {config.DATA_DIR}")
print(f"dataset.csv: {config.DATASET_CSV}")

DATA_DIR: /home/dk/HSE_VKR_DetectingSpeechDefects/data
dataset.csv: /home/dk/HSE_VKR_DetectingSpeechDefects/data/dataset.csv


## Загрузка данных с Яндекс.Диска

In [11]:
# https://github.com/SecFathy/YandexDown/blob/main/YandexCLI.py
import requests
import urllib
import subprocess

class YandexDiskDownloader:
    def __init__(self, link, download_location):
        self.link = link
        self.download_location = Path(download_location)

    def download(self):
        url = f"https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key={urllib.parse.quote(self.link, safe='')}"
        response = requests.get(url)
        response.raise_for_status()
        download_url = response.json()["href"]
        file_name = urllib.parse.unquote(download_url.split("filename=")[1].split("&")[0])
        save_path = self.download_location / file_name
        self.download_location.mkdir(parents=True, exist_ok=True)
        with open(save_path, "wb") as f:
            r = requests.get(download_url, stream=True)
            for chunk in r.iter_content(chunk_size=1024):
                if chunk:
                    f.write(chunk)
                    f.flush()
        print("Download complete.")
        return save_path

downloader = YandexDiskDownloader("https://disk.yandex.ru/d/h1SEmkqBro8Ffg", config.PROJECT_ROOT)
downloader.download()

Download complete.


PosixPath('/home/dk/HSE_VKR_DetectingSpeechDefects/data.tar.gz')

In [12]:
archive = config.PROJECT_ROOT / "data.tar.gz"
if archive.exists():
    config.DATA_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(["tar", "-xzf", str(archive), "-C", str(config.DATA_DIR)], check=True)
    print("Распаковка завершена.")
else:
    print("Файл data.tar.gz не найден.")

Распаковка завершена.


## Удаление дубликатов

Удаляются файлы с "(1)" в имени, если есть оригинал того же размера.

In [13]:
def delete_duplicates(directory_path):
    path = Path(directory_path)
    if not path.exists():
        print(f"Папка {directory_path} не найдена.")
        return
    print(f"Поиск дубликатов в {path.name}...")
    to_delete = []
    for f in path.iterdir():
        if not f.is_file():
            continue
        name = f.name
        if "(1)" not in name:
            continue
        parts = name.rsplit("(1)", 1)
        if len(parts) != 2:
            continue
        original_name = parts[0].rstrip() + parts[1].lstrip()
        original_path = path / original_name
        if original_path.is_file() and original_path.stat().st_size == f.stat().st_size:
            to_delete.append(f)
    for f in to_delete:
        f.unlink()
        print(f"  Удалён дубликат: {f.name}")
    if not to_delete:
        print(f"  Дубликатов не найдено.")

delete_duplicates(config.GOOD_DIR)
delete_duplicates(config.BAD_DIR)

Поиск дубликатов в good...
  Удалён дубликат: eu.468b4fa0-a72e-46e2-800c-b781dd918dc7__стрш(1).wav
  Удалён дубликат: eu.1131c949-c891-44b0-b35f-78fa5be195ae__ст(1).wav
  Удалён дубликат: eu.6056a741-3040-4376-9ad7-f59f83afbace__трлш(1).wav
  Удалён дубликат: eu.67fa1c0b-518e-4155-81a7-0cdcd9432887__сл(1).wav
  Удалён дубликат: eu.47035742-6d06-495b-811f-612d58d6d0e5__срл(1).wav
  Удалён дубликат: eu.7c595ed0-3958-43f3-a29d-3700a9e5c750__чсрлц(1).wav
  Удалён дубликат: eu.3d5af675-092a-4f3e-9265-bdc70bcb2c17__чстр(1).wav
  Удалён дубликат: eu.2b04a3f2-5397-4eea-a93c-ac145a0cbe78__рл(1).wav
  Удалён дубликат: eu.a2480375-3559-4aab-a6da-86a76935afa8__стрш(1).wav
  Удалён дубликат: eu.10e21015-42c9-4ef8-a8c2-66d4057c5423__чстрш(1).wav
Поиск дубликатов в bad...
  Дубликатов не найдено.


## Обнаружение и удаление пустых файлов с помощью VGGish

In [14]:
paths_good = sorted(config.GOOD_DIR.glob("*.wav"))
paths_bad  = sorted(config.BAD_DIR.glob("*.wav"))
paths_all  = [str(p) for p in paths_good] + [str(p) for p in paths_bad]

model = torch.hub.load("harritaylor/torchvggish", "vggish")
model.eval()

bad_files = []
for p in tqdm.tqdm(paths_all):
    try:
        model.forward(p)
    except Exception:
        bad_files.append(p)

empty_set  = set(bad_files)
print(f"Найдено пустых файлов: {len(bad_files)}")
if bad_files:
    for p in bad_files:
        print(f"  {Path(p).name}")
        Path(p).unlink(missing_ok=True)
    print(f"Удалено файлов: {len(bad_files)}")
else:
    print("Нет файлов для удаления.")

Using cache found in /home/dk/.cache/torch/hub/harritaylor_torchvggish_master
100%|██████████| 2782/2782 [00:52<00:00, 52.87it/s]

Найдено пустых файлов: 6
  eu.27f03007-3b1e-4f98-9335-2cb45eb4f0d0__стлц.wav
  eu.429980b8-5979-41c8-847f-21739fcdfb5e__слц.wav
  eu.8803cf6d-1c43-4453-90b4-b8047e599323__тлщ.wav
  eu.904117a9-7971-4722-82cb-a2b9df69cd44__чсрлц.wav
  eu.933eed1a-8629-47bc-8304-7ec5ca65e6a2__стрл.wav
  eu.fc014530-281f-48a6-a8ed-90663a484f1a__трш.wav
Удалено файлов: 6


## Извлечение контрольных букв и длительностей

Буквы берутся из части имени файла после «__». Считаются длительности всех WAV.

In [15]:
RUSSIAN_LOWER = frozenset(
    chr(c) for c in list(range(0x0430, 0x0450)) + [0x0451]
)

def extract_cyrillic_letters(text):
    normalized = unicodedata.normalize("NFC", text)
    return set(c.lower() for c in normalized if c.lower() in RUSSIAN_LOWER)

def fix_utf8_corrupted_on_windows(s):
    out = []
    i = 0
    while i < len(s):
        if i + 1 >= len(s):
            out.append(s[i]); i += 1; continue
        ord1, ord2 = ord(s[i]), ord(s[i+1])
        if ord2 < 0x80 or ord2 > 0xBF:
            if ord1 == 0x0430 and ord2 == 0x041B:
                byte2 = 0xBB
            else:
                out.append(s[i]); i += 1; continue
        else:
            byte2 = ord2
        first_byte = 0xD1 if ord1 == 0x0431 else (0xD0 if ord1 == 0x0430 else None)
        if first_byte is not None:
            try:
                out.append(bytes([first_byte, byte2]).decode("utf-8")); i += 2; continue
            except UnicodeDecodeError:
                pass
        out.append(s[i]); i += 1
    return "".join(out)

def get_name_part_decoded(file_path):
    stem = Path(file_path).stem
    part = stem.split("__", 1)[1] if "__" in stem else stem
    return fix_utf8_corrupted_on_windows(part)

# Актуальные пути после удаления пустых файлов
paths_good_final = sorted(config.GOOD_DIR.glob("*.wav"))
paths_bad_final  = sorted(config.BAD_DIR.glob("*.wav"))
paths_all_final  = [str(p) for p in paths_good_final] + [str(p) for p in paths_bad_final]
labels_all_final = [config.CLASS_GOOD] * len(paths_good_final) + [config.CLASS_BAD] * len(paths_bad_final)

print(f"Файлов после очистки: {len(paths_all_final)} (good={len(paths_good_final)}, bad={len(paths_bad_final)})")

all_letters_set = set()
file_letters = []
for p in paths_all_final:
    name_part = get_name_part_decoded(p)
    letters = extract_cyrillic_letters(name_part)
    if not letters:
        letters = extract_cyrillic_letters(Path(p).stem)
    all_letters_set.update(letters)
    file_letters.append(letters)

all_letters = sorted(all_letters_set)
print(f"Контрольные буквы: {all_letters}")

letter_matrix = np.zeros((len(paths_all_final), len(all_letters)), dtype=np.int32)
for i, letters in enumerate(file_letters):
    for j, letter in enumerate(all_letters):
        letter_matrix[i, j] = int(letter in letters)

print("Считаем длительности...")
durations = np.array([
    len(librosa.load(p, sr=config.TARGET_SR, mono=True)[0]) / config.TARGET_SR
    for p in paths_all_final
])
print(f"Готово. min={durations.min():.2f}s  max={durations.max():.2f}s  mean={durations.mean():.2f}s")

Файлов после очистки: 2776 (good=1875, bad=901)
Контрольные буквы: ['л', 'р', 'с', 'т', 'ц', 'ч', 'ш', 'щ']
Считаем длительности...
Готово. min=0.99s  max=68.66s  mean=10.36s


In [ ]:
# Удаление файлов без речи по порогу длительности
# После прослушивания выяснилось, что три самых коротких файла не содержат речи.

MIN_DURATION_SEC = 1.5

short_mask = durations < MIN_DURATION_SEC
short_paths = [paths_all_final[i] for i in range(len(paths_all_final)) if short_mask[i]]

print(f"Файлов короче {MIN_DURATION_SEC}s: {short_mask.sum()}")
for p in short_paths:
    dur = durations[list(paths_all_final).index(p)]
    print(f"  {Path(p).name}  ({dur:.2f}s)")
    Path(p).unlink(missing_ok=True)

# Фильтрация всех переменных, которые используются при сохранении CSV
keep = ~short_mask
paths_all_final = [paths_all_final[i] for i in range(len(paths_all_final)) if keep[i]]
labels_all_final = [labels_all_final[i] for i in range(len(labels_all_final)) if keep[i]]
file_letters = [file_letters[i] for i in range(len(file_letters)) if keep[i]]
durations = durations[keep]
letter_matrix = letter_matrix[keep]

print(f"Осталось файлов: {len(paths_all_final)}")

Файлов короче 1.5s: 3
  eu.4fd8801a-d7cb-41f6-9080-5a55b4097121__трл.wav  (1.40s)
  eu.5d0a9ee2-f0c7-4f3c-9b13-b661331b2695__рлш.wav  (1.25s)
  eu.8c6aad07-02a0-43e4-9762-2242ab7ba305__трщ.wav  (0.99s)
Осталось файлов: 2773


## Сохранение dataset.csv

In [ ]:
filenames = [Path(p).name for p in paths_all_final]
dirnames  = [Path(p).parent.name for p in paths_all_final]

df_new = pd.DataFrame({
    "filename": filenames,
    "dir": dirnames,
    "label": labels_all_final,
    "duration": durations,
})
for j, letter in enumerate(all_letters):
    df_new[letter] = letter_matrix[:, j]

config.DATA_DIR.mkdir(parents=True, exist_ok=True)
df_new.to_csv(config.DATASET_CSV, index=False, encoding="utf-8")
print(f"Сохранено: {config.DATASET_CSV}")
print(f"Строк: {len(df_new)}, столбцов: {len(df_new.columns)}")
df_new.sample(5)

Сохранено: /home/dk/HSE_VKR_DetectingSpeechDefects/data/dataset.csv
Строк: 2773, столбцов: 12


,filename,dir,label,duration,л,р,с,т,ц,ч,ш,щ
2288,eu.6a3bb499-18e7-4352-abd7-787dfec208a9__стр.wav,bad,1,12.300000,0,1,1,1,0,0,0,0
1834,eu.fab16ce0-a20b-4e11-adb4-92ce55371b3d__тлщ.wav,good,0,6.036812,1,0,0,1,0,0,0,1
1893,eu.047ca7f9-a311-4a95-9d60-88fd41141c0f__трл.wav,bad,1,15.700000,1,1,0,1,0,0,0,0
2335,eu.76869def-c735-479c-9ef2-20abfd42f67e__стр.wav,bad,1,13.320000,0,1,1,1,0,0,0,0
138,eu.14c0ea7d-feb7-4b9e-93fe-33881e0dd697__стр.wav,good,0,10.660000,0,1,1,1,0,0,0,0


## Удаление контрольных букв из имён файлов

Часть `__<буквы>` удаляется из имён WAV-файлов — буквы уже сохранены в CSV.

In [18]:
renamed = 0
skipped = 0

for i, old_path_str in enumerate(paths_all_final):
    old_path = Path(old_path_str)
    stem = old_path.stem
    if '__' not in stem:
        skipped += 1
        continue
    new_stem = stem.split('__', 1)[0]
    new_path = old_path.with_name(new_stem + old_path.suffix)
    if old_path.exists():
        old_path.rename(new_path)
        renamed += 1
    else:
        skipped += 1
    paths_all_final[i] = str(new_path)

print(f"Переименовано: {renamed}, уже без букв (пропущено): {skipped}")

# Обновляем имена файлов в df_new и перезаписываем CSV
df_new['filename'] = [Path(p).name for p in paths_all_final]
df_new.to_csv(config.DATASET_CSV, index=False, encoding='utf-8')
print(f"dataset.csv обновлён: {config.DATASET_CSV}")
df_new[['filename', 'dir', 'label', 'duration']].head(3)

Переименовано: 2773, уже без букв (пропущено): 0
dataset.csv обновлён: /home/dk/HSE_VKR_DetectingSpeechDefects/data/dataset.csv


,filename,dir,label,duration
0,eu.00f3da84-570e-40f5-9949-31cd811ff6ca.wav,good,0,12.415625
1,eu.01062fc6-fbb8-4544-a0f5-2830899a91d0.wav,good,0,6.480000
2,eu.01611791-7a64-42c6-82f4-1f8d46603999.wav,good,0,11.940000
